# verify01: 同一の採用ペアで「painful」と「遷移BERT(質問あり/なし)」を比べる

**ねらい**：前処理・入力・出力は painful とそろえ（全モデルが `dataset/input_pairs.csv` の
**同じ採用ペア**を使う）、**違うのは「BERTの学習・予測のやり方」だけ**にして公平に比較する。

**比べる3モデル**（出力はどれも「ノードの答え 0=はい/1=いいえ/2=不明」の3クラス分類）

| モデル | 学習のしかた | 入力テキスト |
|---|---|---|
| **A. painful** | 痛みノード**だけ**を学習する単体分類器（painfulそのまま） | 採用ペア（質問文なし） |
| **B. 遷移BERT(質問あり)** | 3ノード(痛み/しびれ/振る舞い)を**1本のモデルで共有**学習 | **質問文** + 採用ペア |
| **C. 遷移BERT(質問なし)** | 同上（共有学習） | 採用ペアのみ |

> 採用ペア列の対応：`採用ペア_ひし形_全通り_頭痛_1`=痛み / `_2`=しびれ / `_3`=振る舞い。
> painful は `_1`(痛み) のみを使う。遷移BERTは3つ全部を使う。
> 公平のため **fold分割・ハイパラ(学習率/epoch/batch/max_len)は全モデル共通**。

# 1. セットアップ（GPU環境は git clone / ローカルはそのまま）

In [ ]:
import os, sys, subprocess

REPO_URL = 'https://github.com/enenen13/Emergency_task'
REPO_NAME = 'Emergency_task'
REPO_BRANCH = 'feature/headache-ablation-notebook'  # ノート類・input_pairs.csv があるブランチ
IN_COLAB = 'google.colab' in sys.modules


def _find_repo_root(start):
    d = os.path.abspath(start)
    for _ in range(6):
        if os.path.exists(os.path.join(d, 'dataset', 'input_pairs.csv')):
            return d
        parent = os.path.dirname(d)
        if parent == d:
            break
        d = parent
    return None


REPO_DIR = _find_repo_root(os.getcwd())
cloned = False
if REPO_DIR is None:
    if not os.path.isdir(REPO_NAME):
        print(f'git clone -b {REPO_BRANCH} {REPO_URL} ...')
        subprocess.run(['git', 'clone', '--depth', '1', '-b', REPO_BRANCH, REPO_URL], check=True)
        cloned = True
    REPO_DIR = os.path.abspath(REPO_NAME)
os.chdir(REPO_DIR)
print('REPO_DIR =', REPO_DIR)

if IN_COLAB or cloned:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                    'transformers==4.46.3', 'sentencepiece', 'fugashi',
                    'ipadic', 'unidic-lite', 'protobuf', 'accelerate'], check=True)

CSV_PATH = os.path.join(REPO_DIR, 'dataset', 'input_pairs.csv')
YAML_PATH = os.path.join(REPO_DIR, 'transition_diagram', 'protocol.yaml')
OUT_DIR = os.path.join(REPO_DIR, 'output')
os.makedirs(OUT_DIR, exist_ok=True)
print('CSV :', CSV_PATH, '(exists:', os.path.exists(CSV_PATH), ')')

import torch
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('DEVICE =', DEVICE)

## 1.1 実験設定（全モデル共通＝ここを変えれば3モデルとも同じ条件で変わる）

In [ ]:
# ▼ 3モデルで完全に共通の条件（モデルの違いだけを見るため固定する）
MODEL_NAME = 'cl-tohoku/bert-base-japanese-v3'  # painful と同じベースBERT
MAX_LEN = 256        # painful は512。CPUで重ければ 64 などに下げる
EPOCHS = 3           # CPUで重ければ 1
LR = 2e-5
BATCH = 8            # CPUで重ければ 4
N_FOLDS = 5
USE_STOPWORDS = False  # painful の best(run_052) と同じく除去しない
SEED = 42

import random, numpy as np


def set_seed(seed=SEED):
    """再現性のため乱数を固定（painful と同じ）。"""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


set_seed()
print(f'MAX_LEN={MAX_LEN} EPOCHS={EPOCHS} LR={LR} BATCH={BATCH} N_FOLDS={N_FOLDS} '
      f'USE_STOPWORDS={USE_STOPWORDS}')

# 2. データ読み込みと前処理（painful を流用）

`input_pairs.csv` を読み、painful と同じやり方で **患者ごとに採用ペアを連結**して1テキストにする。
ノードごとに採用ペア列とラベル列を切り替えるだけ。

In [ ]:
import pandas as pd
import yaml

df = pd.read_csv(CSV_PATH)
print('rows:', len(df), '/ patients:', df['id'].nunique())

# ノードの質問文を protocol.yaml から取得（「質問あり」モデル用）
_proto = yaml.safe_load(open(YAML_PATH, encoding='utf-8'))
_h = next(p for p in _proto['protocols'] if p['id'] == 'headache')
_q = {n['id']: n['question'] for n in _h['nodes']
      if 'choices' in n and not n.get('metadata_only', False)}

# 3ノードの定義（採用ペア列 / ラベル列 / 質問文）
NODES = [
    {'key': '痛み',   'adopt': '採用ペア_ひし形_全通り_頭痛_1', 'label': '痛み',   'question': _q['headache_sudden_severe']},
    {'key': 'しびれ', 'adopt': '採用ペア_ひし形_全通り_頭痛_2', 'label': 'しびれ', 'question': _q['headache_numbness_paralysis']},
    {'key': '振る舞い', 'adopt': '採用ペア_ひし形_全通り_頭痛_3', 'label': '振る舞い', 'question': _q['headache_abnormal_behavior']},
]
for n in NODES:
    print(f"  {n['key']}: 質問=「{n['question']}」")

# --- ストップワード除去（painful そのまま。USE_STOPWORDS=True のとき使用）---
import fugashi
_tagger = fugashi.Tagger()
STOPWORD_EXTRA_WORDS = set(['の', 'は', 'を', 'に', 'が', 'で', 'と', 'も', 'から', 'より',
                            'へ', 'や', 'など', 'ので', 'けど', 'けれど', '、', '。', 'です', 'ます'])


def remove_stopwords(text):
    return ''.join(w.surface for w in _tagger(text) if w.surface not in STOPWORD_EXTRA_WORDS)


# --- 患者ごとに「採用ペアを連結した1テキスト」と「ラベル」を作る（painful の input_filtered_function 相当）---
def make_patient_table(adopt_col, label_col, use_question=False, question=''):
    rows = []
    for pid, g in df.groupby('id'):
        pairs = g[g[adopt_col] == True]['ペア'].tolist()
        text = ' '.join(pairs) if pairs else '(発話なし)'
        if USE_STOPWORDS:
            text = remove_stopwords(text)
        if use_question:
            text = question + ' ' + text   # 「質問あり」モデルは先頭に質問文を足す
        rows.append({'id': pid, 'text': text, 'label': int(g[label_col].iloc[0])})
    return pd.DataFrame(rows).set_index('id')


# 動作確認（痛みノード）
_demo = make_patient_table(NODES[0]['adopt'], NODES[0]['label'])
print('\n痛みノードのテキスト例:')
print(' ', _demo['text'].iloc[0][:80])
print('ラベル分布:', _demo['label'].value_counts().sort_index().to_dict())

# 3. ★ここがBERTの中身★（学習と予測。3モデル共通で使う）

ここだけが「入力→出力」の心臓部。**全モデルがこの2つの関数を使う**ので、
モデルの違いは「どんなテキストで学習させるか」だけになる。

In [ ]:
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModelForSequenceClassification

# painful の PainTextDataset をそのまま流用（テキスト→BERT入力トークンに変換）
class PainTextDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_length):
        self.texts = list(texts)
        self.labels = list(labels)
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        enc = self.tokenizer(self.texts[idx], truncation=True, max_length=self.max_length,
                             padding='max_length', return_tensors='pt')
        item = {k: v.squeeze(0) for k, v in enc.items()}
        item['labels'] = torch.tensor(self.labels[idx], dtype=torch.long)
        return item


_tok_cache = {}
def get_tokenizer(name):
    if name not in _tok_cache:
        _tok_cache[name] = AutoTokenizer.from_pretrained(name, trust_remote_code=True)
    return _tok_cache[name]


def build_model(name):
    """3クラス分類のBERTを新しく用意（painful 同様、語彙数ズレは自動補正）。"""
    model = AutoModelForSequenceClassification.from_pretrained(name, num_labels=3)
    tok = get_tokenizer(name)
    if len(tok) != model.config.vocab_size:
        model.resize_token_embeddings(len(tok))
    return model.to(DEVICE)


# ===== 学習：テキストとラベルを渡すと、学習済みBERTを返す =====
def train_model(texts, labels):
    set_seed(SEED)                              # 1) 乱数固定（毎回同じ初期値）
    tokenizer = get_tokenizer(MODEL_NAME)
    model = build_model(MODEL_NAME)             # 2) まっさらな3クラスBERTを用意
    loader = DataLoader(PainTextDataset(texts, labels, tokenizer, MAX_LEN),
                        batch_size=BATCH, shuffle=True)   # 3) 文章を数値に変換しバッチで配る
    optimizer = torch.optim.AdamW(model.parameters(), lr=LR)
    model.train()
    for epoch in range(EPOCHS):                 # 4) EPOCHS回くりかえす
        for batch in loader:
            batch = {k: v.to(DEVICE) for k, v in batch.items()}
            optimizer.zero_grad()
            out = model(**batch)                #    予測する
            out.loss.backward()                 #    正解とのズレ(loss)を逆伝播
            optimizer.step()                    #    重みを少し更新
    return model, tokenizer


# ===== 予測：学習済みBERTにテキストを入れ、一番スコアの高いクラスを返す =====
@torch.no_grad()
def predict_labels(model, tokenizer, texts):
    model.eval()
    preds = []
    ds = PainTextDataset(texts, [0] * len(texts), tokenizer, MAX_LEN)  # ラベルはダミー
    for batch in DataLoader(ds, batch_size=BATCH, shuffle=False):
        batch.pop('labels')
        batch = {k: v.to(DEVICE) for k, v in batch.items()}
        logits = model(**batch).logits
        preds += logits.argmax(dim=-1).cpu().tolist()  # 3クラスのうち最大を選ぶ
    return preds


print('train_model / predict_labels を定義（これが3モデル共通のBERTコア）')

# 4. fold分割（患者単位5-fold・全モデルで同じ分割を使う）

In [ ]:
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score, f1_score

patients = np.array(sorted(df['id'].unique()))
# triage で層化（どの fold にも各トリアージが散るように）
triage = np.array([int(df[df['id'] == p]['トリアージ'].iloc[0]) for p in patients])
skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
FOLDS = [(patients[tr], patients[te]) for tr, te in skf.split(patients, triage)]
print(f'{N_FOLDS}-fold 作成。各foldのtest患者数:', [len(te) for _, te in FOLDS])

# 5. モデルA：painful（痛みノード単体・採用ペアのみ）

painful と同じく **痛みノードだけ**を学習する単体分類器。入力は採用ペア（質問文なし）。

In [ ]:
tabA = make_patient_table(NODES[0]['adopt'], NODES[0]['label'], use_question=False)

accA, f1A = [], []
for i, (tr_ids, te_ids) in enumerate(FOLDS):
    model, tok = train_model(tabA.loc[tr_ids, 'text'].tolist(), tabA.loc[tr_ids, 'label'].tolist())
    yp = predict_labels(model, tok, tabA.loc[te_ids, 'text'].tolist())
    yt = tabA.loc[te_ids, 'label'].tolist()
    accA.append(accuracy_score(yt, yp))
    f1A.append(f1_score(yt, yp, average='macro', zero_division=0))
    print(f'  fold{i}: acc={accA[-1]:.3f} f1={f1A[-1]:.3f}')
    del model
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
print(f'A. painful 痛み: acc={np.mean(accA):.3f}±{np.std(accA):.3f} '
      f'f1={np.mean(f1A):.3f}±{np.std(f1A):.3f}')

# 6. モデルB/C：遷移BERT（3ノード共有・質問あり / 質問なし）

3ノード(痛み/しびれ/振る舞い)の例を**全部まとめて1本のモデル**で学習する（共有）。
`use_question=True` なら入力の先頭にそのノードの質問文を付ける。評価はノードごとに行う。

In [ ]:
def run_transition(use_question):
    # ノードごとの患者別テキスト表を用意
    tabs = {n['key']: make_patient_table(n['adopt'], n['label'], use_question, n['question'])
            for n in NODES}
    per_node = {n['key']: {'acc': [], 'f1': []} for n in NODES}

    for i, (tr_ids, te_ids) in enumerate(FOLDS):
        # --- 学習データ＝3ノード分を全部つなげる（共有学習）---
        train_texts, train_labels = [], []
        for n in NODES:
            t = tabs[n['key']]
            train_texts += t.loc[tr_ids, 'text'].tolist()
            train_labels += t.loc[tr_ids, 'label'].tolist()
        model, tok = train_model(train_texts, train_labels)   # ← 共通コアで1本だけ学習

        # --- 評価はノードごと ---
        line = []
        for n in NODES:
            t = tabs[n['key']]
            yt = t.loc[te_ids, 'label'].tolist()
            yp = predict_labels(model, tok, t.loc[te_ids, 'text'].tolist())
            per_node[n['key']]['acc'].append(accuracy_score(yt, yp))
            per_node[n['key']]['f1'].append(f1_score(yt, yp, average='macro', zero_division=0))
            line.append(f"{n['key']} acc={per_node[n['key']]['acc'][-1]:.3f}")
        print(f'  fold{i}: ' + ' | '.join(line))
        del model
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
    return per_node


print('=== B. 遷移BERT（質問あり）===')
resB = run_transition(use_question=True)
print('=== C. 遷移BERT（質問なし）===')
resC = run_transition(use_question=False)
print('done')

# 7. 結果：ノード別の比較（accuracy / macro-F1, mean±std）

In [ ]:
def ms(vals):
    return f'{np.mean(vals):.3f} ± {np.std(vals):.3f}'


# accuracy 表
rows_acc, rows_f1 = [], []
for n in NODES:
    k = n['key']
    a_painful = ms(accA) if k == '痛み' else '—（painfulは痛みのみ）'
    f_painful = ms(f1A) if k == '痛み' else '—'
    rows_acc.append({'ノード': k, 'A.painful(単体/質問なし)': a_painful,
                     'B.遷移(質問あり)': ms(resB[k]['acc']), 'C.遷移(質問なし)': ms(resC[k]['acc'])})
    rows_f1.append({'ノード': k, 'A.painful(単体/質問なし)': f_painful,
                    'B.遷移(質問あり)': ms(resB[k]['f1']), 'C.遷移(質問なし)': ms(resC[k]['f1'])})

acc_table = pd.DataFrame(rows_acc)
f1_table = pd.DataFrame(rows_f1)
print('===== accuracy =====')
display(acc_table)
print('===== macro-F1 =====')
display(f1_table)

acc_table.to_csv(os.path.join(OUT_DIR, 'verify01_accuracy.csv'), index=False, encoding='utf-8-sig')
f1_table.to_csv(os.path.join(OUT_DIR, 'verify01_f1.csv'), index=False, encoding='utf-8-sig')
print('\nsaved: verify01_accuracy.csv, verify01_f1.csv')
print('参考: painful論文 best run_052(痛み) accuracy=0.716')

# 8. まとめ（読み方）

- **同じ採用ペア・同じfold・同じハイパラ**なので、A/B/Cの差は **「学習のしかた」だけ**から来る。
- **痛みノード**で3つを直接比較できる：
  - A(単体・質問なし) vs C(共有・質問なし) → **単体 vs 共有**の効果。
  - B(共有・質問あり) vs C(共有・質問なし) → **質問文を入れる効果**。
- **しびれ/振る舞い**は B vs C（質問あり/なし）で、共有モデルにおける質問文の効きを見る。
- painful の入力(採用ペア_1)をそのまま使うので、A は painful の再現に相当
  （ハイパラは本ノート共通値。painful原典の best=0.716 は lr5e-5/ep10 などグリッド探索の最良値）。